In [8]:
!pip install streamlit langchain langchain-community pyngrok duckduckgo_search ddgs google-genai --quiet

In [ ]:
%%writefile app.py
import os
import streamlit as st
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from google import genai
from google.genai import types
from google.genai.errors import ServerError

st.set_page_config(page_title="TruthLens Core", page_icon="🔎", layout="wide")

st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Cairo:wght@600;800;900&family=Montserrat:wght@500;700;900&display=swap');
    
    .stApp {
        background: linear-gradient(135deg, #0A0F24 0%, #0F172A 100%);
        color: #E2E8F0;
    }
    
    h1 {
        font-family: 'Montserrat', 'Cairo', sans-serif !important;
        font-weight: 900 !important;
        text-align: center !important;
        background: linear-gradient(45deg, #D2FA65, #38BDF8);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        margin-bottom: 5px !important;
    }
    
    .subtitle {
        text-align: center;
        font-family: 'Montserrat', 'Cairo', sans-serif;
        color: #94A3B8;
        font-size: 18px;
        margin-bottom: 40px;
    }

    div[data-testid="stVerticalBlock"] > div {
        background: rgba(30, 41, 59, 0.4);
        border-radius: 16px;
        padding: 10px;
        border: 1px solid rgba(255, 255, 255, 0.05);
    }

    .stTextArea textarea { 
        font-size: 18px !important; 
        border-radius: 12px !important;
        font-family: 'Cairo', sans-serif;
        background-color: #0F172A !important;
        color: #F8FAFC !important;
        border: 1px solid #334155 !important;
        padding: 15px !important;
    }
    .stTextArea textarea:focus {
        border-color: #D2FA65 !important;
        box-shadow: 0 0 0 1px #D2FA65 !important;
    }
    
    div.stButton > button:first-child {
        background: linear-gradient(90deg, #D2FA65 0%, #B5E836 100%);
        color: #0F172A; 
        font-size: 20px; font-weight: 800; 
        font-family: 'Montserrat', 'Cairo', sans-serif;
        border-radius: 12px; width: 100%; height: 55px; border: none;
        box-shadow: 0 4px 20px rgba(210, 250, 101, 0.2);
        transition: all 0.3s ease;
    }
    div.stButton > button:first-child:hover {
        transform: translateY(-2px);
        box-shadow: 0 6px 25px rgba(210, 250, 101, 0.4);
        background: linear-gradient(90deg, #E3FF91 0%, #D2FA65 100%);
        color: #0F172A;
    }
    
    .stAlert {
        background-color: rgba(15, 23, 42, 0.6) !important;
        border: 1px solid rgba(56, 189, 248, 0.2) !important;
        border-radius: 12px !important;
        color: #E2E8F0 !important;
    }

    .results-card {
        background: rgba(15, 23, 42, 0.8) !important;
        border: 1px solid rgba(210, 250, 101, 0.2) !important;
        border-radius: 16px;
        padding: 30px !important;
        margin-top: 20px;
        font-size: 19px !important;
        line-height: 1.8 !important;
        box-shadow: 0 10px 30px rgba(0,0,0,0.3);
    }
    </style>
""", unsafe_allow_html=True)

api_key = st.secrets["GEMINI_API_KEY"]
client = genai.Client(api_key=api_key, http_options={'api_version': 'v1'})
search_retriever = DuckDuckGoSearchAPIWrapper()

st.markdown("<h1>🔎 TruthLens Core</h1>", unsafe_allow_html=True)
st.markdown("<p class='subtitle'>Automated Global News & Media Content Verification System</p>", unsafe_allow_html=True)

left_space, main_content, right_space = st.columns([1, 4, 1])

with main_content:
    col_input, col_proto = st.columns([2, 1])
    
    with col_input:
        news_input = st.text_area("Enter Content for Verification:", placeholder="Paste context or claim here...", height=180, label_visibility="collapsed")
        submit_button = st.button("Verify Content Now")
    
    with col_proto:
        st.info("""
        **🔎 System Protocol:**
        1. **Context Extraction:** Cross-references global network endpoints.
        2. **Linguistic Profiling:** Assesses vocabulary weight.
        3. **Integrity Valuation:** Matches assertions against factual updates.
        """)

if submit_button and news_input.strip():
    with st.status("🔎 Scanning network and verifying assertions...", expanded=False) as status:
        context_documents = search_retriever.run(news_input)
        input_is_arabic = any("\u0600" <= char <= "\u06FF" for char in news_input[:30])
        
        if input_is_arabic:
            prompt = (
                f"أنت نظام آلي متقدم لتدقيق الحقائق وفحص الأخبار الكاذبة والشائعات. قم بتحليل الادعاء بدقة وفككه بناءً على السياق المرفق فقط.\n\n"
                f"Context: {context_documents}\n\n"
                f"Claim: {news_input}\n\n"
                f"يجب أن تكتب التقرير بالكامل باللغة العربية الفصحى، وبصيغة منسقة ومنظمة جداً تحت هذه العناوين الدقيقة:\n"
                f"1. الحكم النهائي للنظام\n"
                f"2. التدقيق المنطقي والنقدي\n"
                f"3. مطابقة الحقائق المسجلة\n"
                f"4. التناقضات الجوهرية المكتشفة"
            )
        else:
            prompt = (
                f"You are an automated factual auditing system. Verify the claim based strictly on the provided context.\n\n"
                f"Context: {context_documents}\n\n"
                f"Claim: {news_input}\n\n"
                f"Output the report strictly under these exact headers:\n"
                f"1. SYSTEM VERDICT\n"
                f"2. CRITICAL LOGICAL AUDIT\n"
                f"3. REGISTERED FACT COMPARISON\n"
                f"4. CORE DISCREPANCIES DETECTED"
            )
        
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.1)
            )
        except ServerError:
            response = client.models.generate_content(
                model='gemini-1.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.1)
            )
        status.update(label="Analysis Finished!", state="complete")
    
    st.markdown("<br><h3 style='text-align: center; color: #D2FA65;'>📊 Verification Audit Logs</h3>", unsafe_allow_html=True)
    
    with st.container():
        raw_output = response.text
        
        if input_is_arabic:
            st.markdown(f'<div class="results-card" style="direction: rtl; text-align: right;">{raw_output}</div>', unsafe_allow_html=True)
        else:
            st.markdown(f'<div class="results-card" style="text-align: left;">{raw_output}</div>', unsafe_allow_html=True)

Overwriting app.py


In [13]:
import subprocess
from pyngrok import ngrok
ngrok.set_auth_token("3GHU9uzrmbB6QRM340R3Ju9Luie_3Psnbfd62NgWphEE9K5XN")
print("Running Streamlit...")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

public_url = ngrok.connect(8501)
print("\n Link Ready! :", public_url)

Running Streamlit...

 Link Ready! : NgrokTunnel: "https://judgingly-datebook-niece.ngrok-free.dev" -> "http://localhost:8501"




2026-07-09 23:52:49.462 Port 8501 is not available
